The Random Forest + XGBoost + Logistic Regression approach in a stacking classifier will combine the strengths of each model, potentially providing better performance.

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

# Step 1: Load the dataset and preprocess it
# Assuming the dataset "balanced_domain_recommendation_data.csv" exists
df = pd.read_csv("balanced_domain_recommendation_data.csv")

# Encode the 'Domain' target variable
label_encoder = LabelEncoder()
df["Domain"] = label_encoder.fit_transform(df["Domain"])

# Add user preferences as binary features
df["prefers_cloud_computing"] = np.where(df["Domain"] == "Cloud Computing", 1, 0)
df["prefers_data_science"] = np.where(df["Domain"] == "Data Science", 1, 0)
df["prefers_web_development"] = np.where(df["Domain"] == "Web Development", 1, 0)

# Define features (X) and target (y)
X = df[["CGPA", "Hours Spent", "prefers_cloud_computing", "prefers_data_science", "prefers_web_development"]]  # Features
y = df["Domain"]  # Target variable: Domain

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Create base models
# Base models for stacking
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
    ('xgb', xgb.XGBClassifier(n_estimators=1000, max_depth=10, learning_rate=0.01, objective="multi:softmax", num_class=len(label_encoder.classes_), random_state=42)),
    ('svc', LogisticRegression(random_state=42))
]

# Step 3: Meta-model (Logistic Regression)
meta_model = LogisticRegression()

# Step 4: Create the Stacking Classifier
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model)

# Step 5: Train the Stacking Classifier
stacking_clf.fit(X_train, y_train)

# Step 6: Make predictions and evaluate the model
y_pred = stacking_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacking Classifier Accuracy: {accuracy:.2f}")

# Step 7: Save the Stacking Classifier and Label Encoder
joblib.dump(stacking_clf, "stacking_classifier_model_with_preferences.pkl")
joblib.dump(label_encoder, "stacking_label_encoder_with_preferences.pkl")
print("✅ Stacking Classifier and Label Encoder saved.")

# Step 8: Predict the domain based on user input (user preferences and study data)
def recommend_domain_with_preferences(user_input):
    """
    user_input: Dictionary containing user data (CGPA, Hours Spent, and Domain Preference)
    Example:
    {
        'CGPA': <value>, 'Study Hours': <value>, 'Preferred Domain': 'Cloud Computing'
    }
    """
    # Validate user input
    if not isinstance(user_input["CGPA"], (int, float)) or not isinstance(user_input["Study Hours"], (int, float)):
        return "❌ Invalid data type for CGPA or Study Hours. Please enter numeric values."
    
    # Add user preference as features (1 for preference, 0 otherwise)
    user_input_features = {
        "CGPA": user_input["CGPA"],
        "Study Hours": user_input["Study Hours"],
        "prefers_cloud_computing": 1 if user_input["Preferred Domain"] == "Cloud Computing" else 0,
        "prefers_data_science": 1 if user_input["Preferred Domain"] == "Data Science" else 0,
        "prefers_web_development": 1 if user_input["Preferred Domain"] == "Web Development" else 0
    }
    
    input_data = np.array([[user_input_features["CGPA"], user_input_features["Study Hours"],
                            user_input_features["prefers_cloud_computing"], user_input_features["prefers_data_science"],
                            user_input_features["prefers_web_development"]]])
    
    # Predict the domain (encoded)
    predicted_domain_encoded = stacking_clf.predict(input_data)[0]
    
    # Decode the predicted domain using the label encoder
    predicted_domain = label_encoder.inverse_transform([predicted_domain_encoded])[0]
    
    # If the predicted domain is not the preferred domain, suggest both
    if predicted_domain != user_input["Preferred Domain"]:
        return f"We recommend {predicted_domain} based on your CGPA and study hours. However, you prefer {user_input['Preferred Domain']}. Here’s the roadmap for {user_input['Preferred Domain']} as well!"
    else:
        return f"Based on your CGPA, study hours, and preference, we recommend: {predicted_domain}"

# Step 9: Example user input
user_input = {"CGPA": 6.5, "Study Hours": 2, "Preferred Domain": "Data Science"}
domain_recommendation = recommend_domain_with_preferences(user_input)
print(domain_recommendation)


Stacking Classifier Accuracy: 0.81
✅ Stacking Classifier and Label Encoder saved.
We recommend Web Development based on your CGPA and study hours. However, you prefer Data Science. Here’s the roadmap for Data Science as well!


C:\Users\Muqeem\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Muqeem\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


Accuracy of the Meta Model 


In [4]:
# Step 6: Make predictions and evaluate the model
y_pred = stacking_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacking Classifier Accuracy: {accuracy:.2f}")


Stacking Classifier Accuracy: 0.81
